# Propriétés

Une *property* est un mécanisme du langage Python qui permet de contrôler l'accès en lecture, en écriture et en suppression d'un attribut, tout en conservant une syntaxe d'accès simple (sans parenthèses).

Nous allons voir dans ces travaux pratiques comment les utiliser.

## Définition d'une property : le getter

Écrivez une classe `Circle` qui possède :

- un attribut "privé" `_radius`, initialisé dans le constructeur à partir d'un argument `radius`
- une property `radius` qui renvoie la valeur de `_radius`

La property doit pouvoir être lue via `instance.radius`, sans parenthèses.

In [ ]:
class Circle:
  def __init__(self, radius):
    self._radius = radius

  # Votre code ici


c = Circle(5)
print(c.radius)  # Doit valoir 5

### Solution

In [ ]:
class Circle:
  def __init__(self, radius):
    self._radius = radius

  @property
  def radius(self):
    return self._radius


c = Circle(5)
print(c.radius)  # Doit valoir 5

## Syntaxe de property

La syntaxe `@property` est un sucre syntaxique. Sans elle, comment obtenir le même résultat, en utilisant directement la fonction native `property` ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
class Circle:
  def __init__(self, radius):
    self._radius = radius

  def get_radius(self):
    return self._radius

  radius = property(get_radius)


c = Circle(5)
print(c.radius)  # Doit valoir 5

## Ajout d'un setter

Reprenez la classe `Circle` et ajoutez un setter à la property `radius`, qui vérifie que la valeur fournie est strictement positive. Si ce n'est pas le cas, une `ValueError` doit être levée.

Modifiez également le constructeur pour qu'il utilise `self.radius = radius` (et non `self._radius = radius`), afin que la validation s'applique aussi à la construction de l'objet.

In [ ]:
class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  # Votre code ici


c = Circle(5)
print(c.radius)  # Doit valoir 5

c.radius = 10
print(c.radius)  # Doit valoir 10

c.radius = -1  # Doit lever une ValueError

### Solution

In [ ]:
class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("Le rayon doit être strictement positif")
    self._radius = value


c = Circle(5)
print(c.radius)  # 5

c.radius = 10
print(c.radius)  # 10

try:
  c.radius = -1
except ValueError as e:
  print(f"Erreur : {e}")

## Ajout d'un deleter

Ajoutez un deleter à la property `radius`, qui affiche le message `"Suppression de radius"` puis supprime l'attribut `_radius` de l'instance.

In [ ]:
class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("Le rayon doit être strictement positif")
    self._radius = value

  # Votre code ici


c = Circle(5)
del c.radius
print(hasattr(c, "_radius"))  # Doit valoir False

### Solution

In [ ]:
class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("Le rayon doit être strictement positif")
    self._radius = value

  @radius.deleter
  def radius(self):
    print("Suppression de radius")
    del self._radius


c = Circle(5)
del c.radius
print(hasattr(c, "_radius"))  # Doit valoir False

## Property en lecture seule

Ajoutez à la classe `Circle` une property `area`, en lecture seule (sans setter), qui calcule l'aire du disque à partir de `radius` (utilisez `math.pi`).

In [ ]:
import math


class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("Le rayon doit être strictement positif")
    self._radius = value

  # Votre code ici


c = Circle(5)
print(c.area)  # Doit valoir environ 78.53981633974483

Que se passe-t-il si vous exécutez `c.area = 10` ? Essayez avant de lire la solution.

*Votre réponse*

### Solution

Une property sans setter est en lecture seule : toute tentative d'affectation lève une `AttributeError`, car aucun setter n'est défini pour intercepter l'écriture.

En interne, `property` est ce qu'on appelle un *data descriptor* dès qu'il définit `__set__` (même un `__set__` qui se contente de lever une erreur), ce qui lui donne priorité sur le `__dict__` de l'instance.

In [ ]:
import math


class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("Le rayon doit être strictement positif")
    self._radius = value

  @property
  def area(self):
    return math.pi * self._radius ** 2


c = Circle(5)
print(c.area)

try:
  c.area = 10
except AttributeError as e:
  print(f"Erreur : {e}")

## Mise en cache avec `functools.cached_property`

Le calcul de `area` est ici trivial, mais imaginons qu'il s'agisse d'un calcul coûteux (une requête réseau, un calcul complexe...). Il serait dommage de le refaire à chaque accès si `radius` ne change pas entre-temps.

Utilisez [`functools.cached_property`](https://docs.python.org/fr/3/library/functools.html#functools.cached_property) pour ne calculer `area` qu'une seule fois, et vérifiez-le en observant le nombre d'affichages du message de calcul.

In [ ]:
import functools
import math


class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("Le rayon doit être strictement positif")
    self._radius = value

  # Votre code ici : modifiez la property area
  @property
  def area(self):
    print("Calcul de l'aire...")
    return math.pi * self._radius ** 2


c = Circle(5)
print(c.area)
print(c.area)  # Pour l'instant, "Calcul de l'aire..." s'affiche deux fois : il ne doit s'afficher qu'une fois

### Solution

In [ ]:
import functools
import math


class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("Le rayon doit être strictement positif")
    self._radius = value

  @functools.cached_property
  def area(self):
    print("Calcul de l'aire...")
    return math.pi * self._radius ** 2


c = Circle(5)
print(c.area)
print(c.area)  # Le message ne s'affiche qu'une seule fois

Attention, `cached_property` nécessite que les instances possèdent un `__dict__` (c'est le cas par défaut, sauf si la classe définit `__slots__` sans `"__dict__"`), et le cache n'est pas invalidé automatiquement si `radius` change ensuite : il faudrait le gérer manuellement, par exemple en supprimant `self.__dict__["area"]` dans le setter de `radius`.